# Use another model when one is unavailable

Simulate a model outage and let LiteAgents use a fallback. Only the fallback makes a real API call.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/07_model_fallback.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a6/liteagents-0.3.0a6-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Configure a fallback

The demo primary model is intentionally unavailable. In your application, replace it with your preferred model.

In [ ]:
from liteagents import ProfileOptions, RecoveryOptions, run

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/cookbook-unavailable",
    recovery=RecoveryOptions(
        retries={"max_attempts": 1},
        model_fallbacks=["openai/gpt-5.4-mini"],
    ),
)

## 4. Simulate an outage

This function is only for the demo: it fails the primary request before any network call and lets the fallback through.

In [ ]:
from unittest.mock import patch

import litellm

real_completion = litellm.acompletion

async def simulate_outage(*args, model, **kwargs):
    if model.endswith("/cookbook-unavailable"):
        print("Primary model unavailable; trying the fallback.")
        raise litellm.ServiceUnavailableError(
            message="Intentional demo outage", llm_provider="openai", model=model,
        )
    return await real_completion(*args, model=model, **kwargs)

## 5. Run the agent

You should see the outage message followed by **fallback-ready** from the fallback model.

In [ ]:
with patch("litellm.acompletion", side_effect=simulate_outage):
    result = await run("Reply with exactly: fallback-ready", profile=profile)
print(result.text)

For normal use, keep `RecoveryOptions(model_fallbacks=[...])` and call `run()` without the demo patch. Each fallback needs valid provider credentials.

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)